## ACS Youth Baselines — Build Notes
**Author:** Lauren Vo  

### Purpose

Create a **single, clean baseline table at the Census tract level for San Diego County (ACS 5-year, 2023)** that we will use to measure access to youth opportunities. Fields include:

* `pop_total`
* Youth counts & rates: `youth_5_17`, `youth_10_19`, `youth_5_17_per_1k`, `youth_10_19_per_1k`
* `median_hh_income_2023usd`
* Transportation constraint proxy: `households_total`, `households_zero_veh`, `zero_veh_share`

This baseline lets us compute **services per 1,000 youth**, compare across neighborhoods, and start flagging **opportunity deserts**.

---

### Data sources (ACS 5-year, 2023)

**Universe:** Total population / Households (varies by table)
**Geography:** Census **tract** (11-digit GEOID) within San Diego County

| Topic                                | Table ID   | What we use                                                                    |
| ------------------------------------ | ---------- | ------------------------------------------------------------------------------ |
| Sex by Age                           | **B01001** | Total population; age bins that roll up to 5–17 and 10–19 cohorts              |
| Median Household Income              | **B19013** | `B19013_001E` → `median_hh_income_2023usd`                                     |
| Household Size by Vehicles Available | **B08201** | Total households; households with **no vehicle** (to compute `zero_veh_share`) |

**Files in repo**

* Raw CSVs (downloaded from data.census.gov):

  `data/raw/acs5/2023/sexbyage/ACSDT5Y2023.B01001-Data.csv`

  `data/raw/acs5/2023/medianhouseholdincome/ACSDT5Y2023.B19013-Data.csv`

  `data/raw/acs5/2023/vehiclesavailable/ACSDT5Y2023.B08201-Data.csv`
  
  (+ their `Column-Metadata.csv` companions)

---

### Repro steps 

1. **Find repo root** robustly so paths work for everyone.
2. **Load** each table and keep only estimate columns (those ending in `E`) plus geography fields.
3. **Normalize geography**

   * Parse tract **GEOID** from `GEO_ID` (keep 11-digit tracts only).
4. **Build youth cohorts (from B01001)**

   * `youth_5_17 = sum(5–9, 10–14, 15–17; male + female)`
   * `youth_10_19 = sum(10–14, 15–17, 18–19; male + female)`
   * Also compute `youth_*_per_1k = youth_* / pop_total * 1000` (guarded when `pop_total <= 0`).
5. **Income (B19013)**

   * Rename `B19013_001E` → `median_hh_income_2023usd` (numeric).
6. **Zero-vehicle households (B08201)**

   * Extract total households and households with **0 vehicles**; compute `zero_veh_share`.
7. **Join** all fields on `GEOID` (left joins, tract level).
8. **Clean**

   * Drop duplicate name columns, keep valid tracts, sort by `GEOID`.
9. **Write outputs**

   * Raw join: `data/processed/acs_baselines_tract_or_bg_2023.csv`
   * Cleaned table: `data/processed/acs_baselines_tract_2023_clean.csv`

---

### How youth bins map from B01001 (conceptual)

Aggregated the male & female age bins that correspond to:

* **5–17** → `5–9` + `10–14` + `15–17`
* **10–19** → `10–14` + `15–17` + `18–19`
  (Sum both sexes to get the total per cohort.)

> Note: Column names in the CSV are like `B01001_00xE`. The code uses the **column metadata** file to translate those to human-readable age bins, then aggregates.

---

### Output schema (clean file)

`data/processed/acs_baselines_tract_2023_clean.csv`

| Column                                    | Meaning                                     |
| ----------------------------------------- | ------------------------------------------- |
| `GEOID`                                   | 11-digit Census tract ID (string)           |
| `NAME`                                    | Tract label from ACS                        |
| `pop_total`                               | Total population (B01001)                   |
| `youth_5_17`, `youth_10_19`               | Youth counts (constructed from B01001 bins) |
| `youth_5_17_per_1k`, `youth_10_19_per_1k` | Youth per 1,000 residents                   |
| `median_hh_income_2023usd`                | Median household income (B19013)            |
| `households_total`                        | Total households (B08201)                   |
| `households_zero_veh`                     | Households with **0 vehicles** (B08201)     |
| `zero_veh_share`                          | `households_zero_veh / households_total`    |

---

### Assumptions & checks

* Only **tracts** are kept (block groups are filtered out by requiring 11-digit GEOIDs).
* Per-1k rates are set to `NaN` when `pop_total <= 0`.
* Income and counts are coerced to numeric; non-numeric values become `NaN`.
* Sanity checks (row counts, missing GEOIDs) are printed during the run.

---

**Provenance:** All ACS data are from **ACS 2023 5-Year Estimates** (tables: `B01001`, `B19013`, `B08201`) downloaded via data.census.gov and stored under `data/raw/acs5/2023/…` in this repository.


In [ ]:
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    markers = (".git", "pyproject.toml", "Makefile")
    while p.parent != p and not any((p / m).exists() for m in markers):
        p = p.parent
    return p

REPO = find_repo_root()
RAW        = REPO / "data" / "raw"
INTERIM    = REPO / "data" / "interim"
PROCESSED  = REPO / "data" / "processed"
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

ACS        = RAW / "acs5" / "2023"
SEXBYAGE   = ACS / "sexbyage"              / "ACSDT5Y2023.B01001-Data.csv"
SEXBYAGE_MD= ACS / "sexbyage"              / "ACSDT5Y2023.B01001-Column-Metadata.csv"
INCOME     = ACS / "medianhouseholdincome" / "ACSDT5Y2023.B19013-Data.csv"
INCOME_MD  = ACS / "medianhouseholdincome" / "ACSDT5Y2023.B19013-Column-Metadata.csv"
VEH        = ACS / "vehiclesavailable"     / "ACSDT5Y2023.B08201-Data.csv"
VEH_MD     = ACS / "vehiclesavailable"     / "ACSDT5Y2023.B08201-Column-Metadata.csv"

print("Repo root:", REPO)
for p in [SEXBYAGE, SEXBYAGE_MD, INCOME, INCOME_MD, VEH, VEH_MD]:
    print(p.relative_to(REPO), "->", p.exists())

Repo root: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts
data/raw/acs5/2023/sexbyage/ACSDT5Y2023.B01001-Data.csv -> True
data/raw/acs5/2023/sexbyage/ACSDT5Y2023.B01001-Column-Metadata.csv -> True
data/raw/acs5/2023/medianhouseholdincome/ACSDT5Y2023.B19013-Data.csv -> True
data/raw/acs5/2023/medianhouseholdincome/ACSDT5Y2023.B19013-Column-Metadata.csv -> True
data/raw/acs5/2023/vehiclesavailable/ACSDT5Y2023.B08201-Data.csv -> True
data/raw/acs5/2023/vehiclesavailable/ACSDT5Y2023.B08201-Column-Metadata.csv -> True


In [6]:
import pandas as pd
import numpy as np

def read_csv_safe(path): 
    return pd.read_csv(path, dtype=str, low_memory=False)

def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def extract_geoid(geo_id: pd.Series) -> pd.Series:
    return geo_id.str.split("US", n=1).str[-1]

def keep_estimates(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["NAME", "GEO_ID"] + [c for c in df.columns if c.endswith("E")]
    return df.loc[:, [c for c in cols if c in df.columns]]

In [ ]:
b01001 = keep_estimates(read_csv_safe(SEXBYAGE))
b19013 = keep_estimates(read_csv_safe(INCOME))
b08201 = keep_estimates(read_csv_safe(VEH))

for t in (b01001, b19013, b08201):
    t["GEOID"] = extract_geoid(t["GEO_ID"])

In [ ]:
# B01001 codes
B01001_TOTAL = "B01001_001E"
M_5_9, M_10_14, M_15_17, M_18_19 = "B01001_004E","B01001_005E","B01001_006E","B01001_007E"
F_5_9, F_10_14, F_15_17, F_18_19 = "B01001_028E","B01001_029E","B01001_030E","B01001_031E"

needed = [B01001_TOTAL, M_5_9, M_10_14, M_15_17, M_18_19, F_5_9, F_10_14, F_15_17, F_18_19]
for c in needed:
    if c not in b01001.columns:
        b01001[c] = "0"
    b01001[c] = to_num(b01001[c])

b01001_clean = b01001[["GEOID","NAME", *needed]].copy()
b01001_clean.rename(columns={B01001_TOTAL:"pop_total"}, inplace=True)

b01001_clean["youth_5_17"]  = (b01001_clean[M_5_9]  + b01001_clean[M_10_14] + b01001_clean[M_15_17]
                              + b01001_clean[F_5_9]  + b01001_clean[F_10_14] + b01001_clean[F_15_17])

b01001_clean["youth_10_19"] = (b01001_clean[M_10_14] + b01001_clean[M_15_17] + b01001_clean[M_18_19]
                              + b01001_clean[F_10_14] + b01001_clean[F_15_17] + b01001_clean[F_18_19])

b01001_clean["youth_5_17_perc"]  = 100 * b01001_clean["youth_5_17"]  / b01001_clean["pop_total"]
b01001_clean["youth_10_19_perc"] = 100 * b01001_clean["youth_10_19"] / b01001_clean["pop_total"]

b01001_clean.head(3)

,GEOID,NAME,NAME,pop_total,B01001_004E,B01001_005E,B01001_006E,B01001_007E,B01001_028E,B01001_029E,B01001_030E,B01001_031E,youth_5_17,youth_10_19,youth_5_17_perc,youth_10_19_perc
0,Geography,Geographic Area Name,Geographic Area Name,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,06073000100,Census Tract 1; San Diego County; California,Census Tract 1; San Diego County; California,2746.0,46.0,90.0,86.0,22.0,47.0,80.0,11.0,21.0,360.0,310.0,13.109978,11.289148
2,06073000201,Census Tract 2.01; San Diego County; California,Census Tract 2.01; San Diego County; California,2376.0,31.0,102.0,53.0,0.0,45.0,74.0,50.0,0.0,355.0,279.0,14.941077,11.742424


In [ ]:
if "B19013_001E" not in b19013.columns:
    raise ValueError("B19013_001E missing from income table.")

b19013_clean = b19013[["GEOID","NAME","B19013_001E"]].copy()
b19013_clean.rename(columns={"B19013_001E":"median_hh_income_2023usd"}, inplace=True)
b19013_clean["median_hh_income_2023usd"] = to_num(b19013_clean["median_hh_income_2023usd"])
b19013_clean.head(3)

,GEOID,NAME,NAME,median_hh_income_2023usd
0,Geography,Geographic Area Name,Geographic Area Name,NaN
1,06073000100,Census Tract 1; San Diego County; California,Census Tract 1; San Diego County; California,211023.0
2,06073000201,Census Tract 2.01; San Diego County; California,Census Tract 2.01; San Diego County; California,110476.0


In [ ]:
veh = b08201.copy()

if "B08201_001E" not in veh.columns:
    raise ValueError("B08201_001E (total households) missing.")

veh["households_total"] = to_num(veh["B08201_001E"])

if "B08201_002E" in veh.columns:
    veh["households_zero_veh"] = to_num(veh["B08201_002E"])
else:
    md = read_csv_safe(VEH_MD)
    md.columns = [c.strip().lower() for c in md.columns]
    code_col  = next(c for c in md.columns if "name"  in c)   
    label_col = next(c for c in md.columns if "label" in c)   
    codes = (
        md.loc[md[label_col].str.contains("No vehicle available", case=False, na=False), code_col]
        .str.strip().tolist()
    )
    codes = [c for c in codes if c.endswith("_E") and c in veh.columns]
    if not codes:
        raise ValueError("Couldn't identify 'No vehicle available' columns from metadata.")
    veh["households_zero_veh"] = to_num(veh[codes].astype(float).sum(axis=1))

veh_clean = veh[["GEOID","NAME","households_total","households_zero_veh"]].copy()
veh_clean["zero_veh_share"] = veh_clean["households_zero_veh"] / veh_clean["households_total"]
veh_clean.head(3)

,GEOID,NAME,NAME,households_total,households_zero_veh,zero_veh_share
0,Geography,Geographic Area Name,Geographic Area Name,NaN,NaN,NaN
1,06073000100,Census Tract 1; San Diego County; California,Census Tract 1; San Diego County; California,1103.0,10.0,0.009066
2,06073000201,Census Tract 2.01; San Diego County; California,Census Tract 2.01; San Diego County; California,1258.0,155.0,0.123211


In [11]:
acs = (
    b01001_clean.merge(
        b19013_clean[["GEOID","median_hh_income_2023usd"]],
        on="GEOID", how="left"
    )
    .merge(
        veh_clean[["GEOID","households_total","households_zero_veh","zero_veh_share"]],
        on="GEOID", how="left"
    )
)

acs["youth_5_17_per_1k"]  = 1000 * acs["youth_5_17"]  / acs["pop_total"]
acs["youth_10_19_per_1k"] = 1000 * acs["youth_10_19"] / acs["pop_total"]

keep = [
    "GEOID","NAME","pop_total",
    "youth_5_17","youth_10_19","youth_5_17_per_1k","youth_10_19_per_1k",
    "median_hh_income_2023usd",
    "households_total","households_zero_veh","zero_veh_share"
]
acs_final = acs[keep].sort_values("GEOID").reset_index(drop=True)

out_csv = PROCESSED / "acs_baselines_tract_or_bg_2023.csv"
acs_final.to_csv(out_csv, index=False)
out_csv, acs_final.shape


(PosixPath('/Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/processed/acs_baselines_tract_or_bg_2023.csv'),
 (738, 12))

In [12]:
display(acs_final.head(10))
acs_final[["youth_5_17_per_1k","youth_10_19_per_1k","median_hh_income_2023usd","zero_veh_share"]].describe()


,GEOID,NAME,NAME,pop_total,youth_5_17,youth_10_19,youth_5_17_per_1k,youth_10_19_per_1k,median_hh_income_2023usd,households_total,households_zero_veh,zero_veh_share
0,06073000100,Census Tract 1; San Diego County; California,Census Tract 1; San Diego County; California,2746.0,360.0,310.0,131.099782,112.891479,211023.0,1103.0,10.0,0.009066
1,06073000201,Census Tract 2.01; San Diego County; California,Census Tract 2.01; San Diego County; California,2376.0,355.0,279.0,149.410774,117.424242,110476.0,1258.0,155.0,0.123211
2,06073000202,Census Tract 2.02; San Diego County; California,Census Tract 2.02; San Diego County; California,4019.0,344.0,247.0,85.593431,61.458074,121264.0,2253.0,90.0,0.039947
3,06073000301,Census Tract 3.01; San Diego County; California,Census Tract 3.01; San Diego County; California,2487.0,92.0,111.0,36.992360,44.632087,79605.0,1368.0,123.0,0.089912
4,06073000302,Census Tract 3.02; San Diego County; California,Census Tract 3.02; San Diego County; California,2925.0,85.0,41.0,29.059829,14.017094,89850.0,1773.0,79.0,0.044557
5,06073000400,Census Tract 4; San Diego County; California,Census Tract 4; San Diego County; California,4079.0,121.0,91.0,29.664133,22.309390,89565.0,2509.0,340.0,0.135512
6,06073000500,Census Tract 5; San Diego County; California,Census Tract 5; San Diego County; California,2828.0,234.0,161.0,82.743989,56.930693,101875.0,1572.0,139.0,0.088422
7,06073000600,Census Tract 6; San Diego County; California,Census Tract 6; San Diego County; California,3111.0,61.0,56.0,19.607843,18.000643,99161.0,1879.0,174.0,0.092602
8,06073000700,Census Tract 7; San Diego County; California,Census Tract 7; San Diego County; California,4371.0,75.0,25.0,17.158545,5.719515,109484.0,2587.0,260.0,0.100503
9,06073000800,Census Tract 8; San Diego County; California,Census Tract 8; San Diego County; California,5105.0,225.0,107.0,44.074437,20.959843,92348.0,2898.0,139.0,0.047964


,youth_5_17_per_1k,youth_10_19_per_1k,median_hh_income_2023usd,zero_veh_share
count,735.000000,735.000000,724.000000,732.000000
mean,148.527286,119.281729,109939.169890,0.055036
std,60.066114,59.520336,42330.828767,0.065431
min,0.000000,0.000000,24125.000000,0.000000
25%,116.088617,85.907905,78217.250000,0.016225
50%,155.036095,117.724519,104273.500000,0.036865
75%,187.048111,146.556075,133967.000000,0.075102
max,338.547747,688.869153,246078.000000,1.000000


In [ ]:
fp = PROCESSED / "acs_baselines_tract_or_bg_2023.csv"
acs = pd.read_csv(fp, dtype={"GEOID":"string"})

if "NAME.1" in acs.columns:
    acs = acs.drop(columns=["NAME.1"])

mask_geoid_ok = acs["GEOID"].str.len().eq(11) & acs["GEOID"].str.isnumeric()
acs = acs.loc[mask_geoid_ok].copy()

for col_num, outcol in [
    ("youth_5_17",  "youth_5_17_per_1k"),
    ("youth_10_19", "youth_10_19_per_1k"),
]:
    bad = (acs["pop_total"] <= 0) | acs["pop_total"].isna()
    acs.loc[bad, outcol] = pd.NA

acs = acs.sort_values("GEOID").reset_index(drop=True)

out = PROCESSED / "acs_baselines_tract_2023_clean.csv"
acs.to_csv(out, index=False)
print("Wrote:", out, "rows:", len(acs))

preview = pd.read_csv(out, dtype={"GEOID":"string"})
print("Columns:", list(preview.columns))
display(preview.head(10)) 

Wrote: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/processed/acs_baselines_tract_2023_clean.csv rows: 737
Columns: ['GEOID', 'NAME', 'pop_total', 'youth_5_17', 'youth_10_19', 'youth_5_17_per_1k', 'youth_10_19_per_1k', 'median_hh_income_2023usd', 'households_total', 'households_zero_veh', 'zero_veh_share']


,GEOID,NAME,pop_total,youth_5_17,youth_10_19,youth_5_17_per_1k,youth_10_19_per_1k,median_hh_income_2023usd,households_total,households_zero_veh,zero_veh_share
0,06073000100,Census Tract 1; San Diego County; California,2746.0,360.0,310.0,131.099782,112.891479,211023.0,1103.0,10.0,0.009066
1,06073000201,Census Tract 2.01; San Diego County; California,2376.0,355.0,279.0,149.410774,117.424242,110476.0,1258.0,155.0,0.123211
2,06073000202,Census Tract 2.02; San Diego County; California,4019.0,344.0,247.0,85.593431,61.458074,121264.0,2253.0,90.0,0.039947
3,06073000301,Census Tract 3.01; San Diego County; California,2487.0,92.0,111.0,36.992360,44.632087,79605.0,1368.0,123.0,0.089912
4,06073000302,Census Tract 3.02; San Diego County; California,2925.0,85.0,41.0,29.059829,14.017094,89850.0,1773.0,79.0,0.044557
5,06073000400,Census Tract 4; San Diego County; California,4079.0,121.0,91.0,29.664133,22.309390,89565.0,2509.0,340.0,0.135512
6,06073000500,Census Tract 5; San Diego County; California,2828.0,234.0,161.0,82.743989,56.930693,101875.0,1572.0,139.0,0.088422
7,06073000600,Census Tract 6; San Diego County; California,3111.0,61.0,56.0,19.607843,18.000643,99161.0,1879.0,174.0,0.092602
8,06073000700,Census Tract 7; San Diego County; California,4371.0,75.0,25.0,17.158545,5.719515,109484.0,2587.0,260.0,0.100503
9,06073000800,Census Tract 8; San Diego County; California,5105.0,225.0,107.0,44.074437,20.959843,92348.0,2898.0,139.0,0.047964
